# Historical Weather Silver Processing

Read data from the Bronze Delta table, apply cleansing and standardization, and load the refined data into the Silver Delta table for the Urban Mobility platform.

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.table("urban_mobility.bronze.historical_weather")

display(df)

In [0]:
df.printSchema()

In [0]:
silver_df = (
    df
    .select(
        "latitude",
        "longitude",
        "elevation",
        "timezone",
        "timezone_abbreviation",
        "utc_offset_seconds",
        F.posexplode("hourly.time").alias("pos", "time"),
        "hourly.temperature_2m",
        "hourly.precipitation",
        "hourly.wind_speed_10m"
    )
    .select(
        "latitude",
        "longitude",
        "elevation",
        "timezone",
        "timezone_abbreviation",
        "utc_offset_seconds",
        "time",
        F.col("temperature_2m")[F.col("pos")].alias("temperature_2m"),
        F.col("precipitation")[F.col("pos")].alias("precipitation"),
        F.col("wind_speed_10m")[F.col("pos")].alias("wind_speed_10m")
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "time",
    F.to_timestamp("time")
)

In [0]:
display(silver_df)

In [0]:
(
    silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("urban_mobility.silver.historical_weather")
)

In [0]:
display(spark.table("urban_mobility.silver.historical_weather"))